import pandas as pd
from functools import reduce
import psycopg2 as pg
import pandas as pd
from datetime import datetime, timedelta
import sqlalchemy
import datetime as dt
import numpy as np
pd.set_option('display.max_rows',2000)
pd.set_option('display.max_columns',999)
import smtplib
import re
from sqlalchemy import Table, Column, Integer, String, MetaData, ForeignKey, create_engine, select, inspect, and_, or_, desc
import gspread
import gspread_dataframe as gd
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from IPython.core.display import display, HTML
from premailer import transform
import datetime as dt
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import time as tm
import sqlalchemy
import warnings
warnings.filterwarnings("ignore")
import pandas.io.sql as psql
from tqdm import tqdm
import calendar
import os
import glob
import requests
from pandas.io.json import json_normalize
import requests
from bs4 import BeautifulSoup
import datetime as dt
from datetime import datetime, timedelta
import json
from tqdm import tqdm
from PIL import Image
import time as tm
import requests
import random
import json
import urllib.request
from PIL import Image

from googleapiclient import discovery
from apiclient.http import MediaFileUpload
import gspread
import gspread_dataframe as gd
from oauth2client.service_account import ServiceAccountCredentials 

In [ ]:
import csv
import json
import hashlib
import requests
import datetime
#from humanfriendly import format_timespan
import pandas as pd
#from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
import json
#from pymongo import MongoClient
#import pymysql
from datetime import datetime,timedelta
#import matplotlib.pyplot as plt
import ast
import psycopg2


In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns', 5000)
pd.set_option('display.width', 100000)



In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


In [ ]:
def fetch_veh_image(user_code):
    url = DOC_STORE_API_BASE+"/documents/byentityids"
    payload="{\n    \"entityType\": \"OPERATOR\",\n    \"entityIds\": [ \""+ str(user_code) +"\"] ,\n    \"documentTypes\": [\n        \"AADHAAR_CARD\"    ]\n}"
    headers = {
          'Content-Type': 'application/json'
        } 
    response = requests.request("POST", url, headers=headers, data = payload)
    json_dat = json.loads(response.text)
    return json_dat

In [ ]:
def check_aadhar_DL_front(aad):
    url = VISION_API_BASE+"/ocr/v1/detect"
    payload = json.dumps({
      "s3Bucket": "example-bucket",
      "s3Key": aad,
      "search": [
        {
          "key": "Government",
          "value": "Government"
        },
        {
          "key": "Election",
          "value": "Commission"
        },  
        {
          "key": "Driving",
          "value": "Licence"
        }            
      ],
      "ignoreTextCaseWhileSearch": True
    })
    headers = {
      'Content-Type': 'application/json'
    }
    response = requests.request("POST", url, headers=headers, data = payload)
    res = json.loads(response.text)
    final_response = json_normalize(res['data']['detections'][0])
    final_r = final_response['success'].iloc[0]
    final_response1 = json_normalize(res['data']['detections'][1])
    final_r1 = final_response1['success'].iloc[0]
    final_response2 = json_normalize(res['data']['detections'][2])
    final_r2 = final_response2['success'].iloc[0]
    if final_r1 == True:
        rvt = 'Voter'
    if final_r2 == True:
        rvt = 'Driving'
    if final_r == True:
        rvt = 'Aadhar'
    return rvt

In [ ]:
def check_aadhar_DL_back(aad):
    url = VISION_API_BASE+"/ocr/v1/detect"
    payload = json.dumps({
      "s3Bucket": "example-bucket",
      "s3Key": aad,
      "search": [
        {
          "key": "Unique",
          "value": "Identification"
        },
        {
          "key": "Election",
          "value": "Commission"
        },  
        {
          "key": "Driving",
          "value": "Licence"
        }            
      ],
      "ignoreTextCaseWhileSearch": True
    })
    headers = {
      'Content-Type': 'application/json'
    }
    response = requests.request("POST", url, headers=headers, data = payload)
    res = json.loads(response.text)
    final_response = json_normalize(res['data']['detections'][0])
    final_r = final_response['success'].iloc[0]
    final_response1 = json_normalize(res['data']['detections'][1])
    final_r1 = final_response1['success'].iloc[0]
    final_response2 = json_normalize(res['data']['detections'][2])
    final_r2 = final_response2['success'].iloc[0]
    if final_r1 == True:
        rvt = 'Voter'
    if final_r2 == True:
        rvt = 'Driving'
    if final_r == True:
        rvt = 'Aadhar'
    return rvt

In [ ]:
def check_doc(user_code):
    try:
        json_dat = fetch_veh_image(user_code)
        norm_json = json_normalize(json_dat['data']['documents'][0])
        df = norm_json
        front_img = df['frontImgUrl'][0]
        f1 = front_img.split('amazonaws.com/')
        f2 = f1[1].split('?X-Amz-Security-Token')
        front_image_url = f2[0]
        back_image = df['backImgUrl'][0]
        b1 = back_image.split('amazonaws.com/')
        b2 = b1[1].split('?X-Amz-Security-Token')
        back_image_url = b2[0]
    except:
        pass
    value1 = ''
    value2 = ''
    try:
        value1 = check_aadhar_DL_front(front_image_url)
    except:
        pass
    try:
        value2 = check_aadhar_DL_back(back_image_url)
    except:
        pass
    final = [user_code,value1 + value2]
    print([user_code,value1 + value2])
    return final

ws = gc.open_by_url('GOOGLE_SHEET_URL').worksheet('Data')
#gd.set_with_dataframe(ws,df4,resize=False)
data = pd.DataFrame(ws.get_all_records())

In [ ]:
data1=pd.read_csv('Empty - additional_data_for_26k_set2.csv')

In [ ]:
data1.rename(columns={'User_code':'user_code'},inplace=True)

In [ ]:
pd.DataFrame().to_csv('kyc_ocr_doc_validation.csv')

In [ ]:
from multiprocessing.dummy import Pool as ThreadPool

batch_count=100

last_run_data=pd.read_csv('kyc_ocr_doc_validation.csv',error_bad_lines=False)

last_run_count=len(last_run_data)-batch_count

print('RE-STARTING FROM : ****',last_run_count)

for i in range (last_run_count,len(data1)+batch_count,batch_count):
    print("**** CURRENTLY AT - ",i," ******")
    pool = ThreadPool(20)
    reuslts=pool.map(check_doc,data1['user_code'][i:i+batch_count])
    data=pd.DataFrame(reuslts)
    data.to_csv('kyc_ocr_doc_validation.csv', mode='a', index=False)    
    pool.close()
    pool.join()

In [ ]:
data

In [ ]:
pd.read_csv('kyc_ocr_doc_validation.csv',error_bad_lines=False)